In [2]:
# --- CONFIGURATION ---
URI = "bolt://localhost:7687"  # Direct local connection
AUTH = ("neo4j", "VzZ@BTLg@JBoG94NzPJg9VJeTHaSRyk2") 
DB_NAME = "neo4j"

In [5]:
# Example complete graph pipeline
import ollama
import time
import json
import re
from neo4j import GraphDatabase

# --- CONFIGURATION ---
EMBED_MODEL = "mxbai-embed-large"
LLM_MODEL = "qwen2.5-coder"

driver = GraphDatabase.driver(URI, auth=AUTH)

class GraphRAGPipeline:
    def __init__(self):
        self.driver = driver

    def log(self, stage, message, data=None):
        print(f"\n{'='*20} [DEBUG: {stage.upper()}] {'='*20}")
        print(message)
        if data:
            print(f"\n--- DATA ({stage}) ---")
            if isinstance(data, (dict, list)):
                print(json.dumps(data, indent=2, default=str))
            else:
                print(data)
        print("="*60)

    def get_system_context(self):
        """
        Retrieves the LIVE database state with grouped schema, 
        truncated property examples, and vector index samples.
        """
        with self.driver.session() as session:
            # 1. Grouped Graph Structure
            schema_query = """
            CALL apoc.meta.graph() YIELD nodes, relationships
            UNWIND relationships AS rel
            WITH startNode(rel).name AS source, type(rel) AS rel_type, endNode(rel).name AS target
            RETURN source, rel_type, target ORDER BY source
            """
            schema_data = session.run(schema_query).data()
            formatted_schema = [f"(:{r['source']})-[:{r['rel_type']}]->(:{r['target']})" for r in schema_data]
    
            # 2. Properties with Examples
            props_query = """
            CALL apoc.meta.nodeTypeProperties() 
            YIELD nodeLabels, propertyName, propertyTypes 
            WITH nodeLabels[0] AS label, propertyName, propertyTypes[0] AS type
            WHERE NOT propertyName IN ['embedding', 'embedding_summary']
            ORDER BY label, propertyName
            
            // Find a non-null sample for each specific property
            CALL (label, propertyName) {
                MATCH (n) 
                WHERE label IN labels(n) AND n[propertyName] IS NOT NULL
                RETURN n[propertyName] AS sample_value 
                LIMIT 1
            }
            
            RETURN label, propertyName, type, sample_value
            """
            props_data = session.run(props_query).data()
            
            props_dict = {}
            for p in props_data:
                label = p['label']
                if label not in props_dict:
                    props_dict[label] = []
                
                val = p['sample_value']
                # Truncate strings to 50 chars, otherwise stringify
                if isinstance(val, str):
                    example = f"'{val[:50]}...'" if len(val) > 50 else f"'{val}'"
                else:
                    example = str(val)
                    
                props_dict[label].append(f"\n  - {p['propertyName']} ({p['type']}): {example}")
    
            # 3. Vector Indexes with 'embedding_summary' samples
            index_query = """
            SHOW INDEXES YIELD name, type, labelsOrTypes 
            WHERE type = 'VECTOR' 
            RETURN name, labelsOrTypes[0] AS label
            """
            index_records = session.run(index_query).data()
            
            formatted_indexes = []
            for idx in index_records:
                label = idx['label']
                # Corrected: IS NOT NULL to find a real sample
                summary_sample = session.run(
                    f"MATCH (n:{label}) WHERE n.embedding_summary IS NOT NULL RETURN n.embedding_summary LIMIT 1"
                ).single()
                
                sample_text = summary_sample[0] if summary_sample else "No summary available"
                formatted_indexes.append(
                    f"Index: {idx['name']} (Label: {label})\n"
                    f"   -> Sample embedding_summary: \"{sample_text[:200]}...\""
                )
    
        # Building the Final Markdown String
        context = ["### LIVE DATABASE SCHEMA (LLM CONTEXT GUIDE)"]
        context.append("\n**1. GRAPH STRUCTURE (Grouped by Source):**")
        context.extend(list(dict.fromkeys(formatted_schema))) # Remove duplicates and keep order
        
        context.append("\n**2. NODE PROPERTIES & EXAMPLES:**")
        for label, properties in props_dict.items():
            context.append(f"- **{label}**: {'; '.join(properties)}")
            
        context.append("\n**3. VECTOR INDEXES (Use for semantic search):**")
        context.extend(formatted_indexes)
    
        return "\n".join(context)
        
    def get_embedding(self, text):
        res = ollama.embeddings(model=EMBED_MODEL, prompt=text)
        return res['embedding']

    def generate_cypher_query(self, user_query, schema_context):
        prompt = f"""
        Objective: Generate a simple, valid Neo4j Cypher query.

        SCHEMA CONTEXT:
        {schema_context}

        USER QUESTION: "{user_query}"
        PARAMETER: $embedding (Vector representation of the user question)

        CRITICAL INSTRUCTIONS:
        1. **Do Not Overthink**: Use standard Cypher (`MATCH`, `WHERE`, `RETURN`). Do NOT use GDS or complex graph algorithms unless explicitly requested.
        2. **Utilize Embeddings**: Look at the "VECTOR INDEXES" in the schema. Use `db.index.vector.queryNodes` to find nodes that match the user's intent semantically (e.g., matching "devs" or "certifications" via their respective indexes).
        3. **Schema First**: Use ONLY the labels, properties, and relationship types provided in the SCHEMA CONTEXT.
        4. **Simplicity**: If you can solve the query with a vector lookup followed by a simple relationship match, do that. 

        RETURN ONLY THE CYPHER CODE. NO MARKDOWN. NO EXPLANATIONS.
        """
        
        self.log("Query Generation", "Prompting LLM...")
        res = ollama.generate(model=LLM_MODEL, prompt=prompt)
        
        # Clean response
        cypher = re.sub(r"```cypher|```", "", res['response']).strip()
        self.log("Query Generation", f"Generated Cypher:\n{cypher}")
        return cypher

    def execute_query(self, cypher, embedding):
        with self.driver.session() as session:
            try:
                result = session.run(cypher, embedding=embedding)
                return result.data()
            except Exception as e:
                return [f"Cypher Error: {str(e)}"]

    def generate_final_answer(self, user_query, db_data):
        """
        Step 3: Synthesis of the final answer.
        """
        prompt = f"""
        You are a Data Analyst Assistant for a GraphRAG system.
        
        Your task is to answer the User's Question based STRICTLY and ONLY on the provided 'Retrieved Data' from our Neo4j database.
        
        --------------------------------------------------
        CONTEXT:
        User Question: "{user_query}"
        
        Retrieved Data (JSON from Neo4j):
        {json.dumps(db_data, indent=2)}
        --------------------------------------------------
        
        INSTRUCTIONS:
        
        1. **STRICT DATA GROUNDING**: 
           - You must answer the question using *only* the facts present in the 'Retrieved Data'.
           - Do NOT use your own internal knowledge or general assumptions.
           - If the data says "Count: 5", say "There are 5...". If the data lists specific names, list them.
        
        2. **HANDLE EMPTY/IRRELEVANT DATA**:
           - If 'Retrieved Data' is empty (`[]`) or contains no relevant information to the question, you MUST explicitly state: 
             "I could not find any data in the database matching your request."
           - Do NOT attempt to answer the question generally (e.g., do NOT say "Typically, Data Scientists need Python..."). 
           - You may politely suggest the user check their spelling or try a broader search term.
        
        3. **FORMATTING**:
           - Provide a clear, concise, and natural response.
           - If the data contains a list (e.g., certifications, career steps), use bullet points.
           - Do not mention technical terms like "JSON", "Cypher", "Neo4j", or "Vector Search" in the final output. Speak as a helpful assistant.
           - If the data implies a progression (e.g., "Job A -> Job B"), describe it as a career path.
        
        4. **TONE**:
           - Professional, objective, and direct.
        """
        
        self.log("Synthesis", "Generating final response with strict data grounding...", prompt)
        res = ollama.generate(model=LLM_MODEL, prompt=prompt)
        return res['response']

    def run(self, user_query):
        print(f"\n--- Processing: {user_query} ---")
        context = self.get_system_context()
        vector = self.get_embedding(user_query)
        cypher = self.generate_cypher_query(user_query, context)
        data = self.execute_query(cypher, vector)
        self.log("Execution", "Context", context)
        answer = self.generate_final_answer(user_query, data)
        print(f"\n[ANSWER]: {answer}")

pipeline = GraphRAGPipeline()
pipeline.run("How many devs have a Project Management certification?")


--- Processing: How many devs have a Project Management certification? ---

==================== [DEBUG: QUERY GENERATION] ====================
Prompting LLM...

==================== [DEBUG: QUERY GENERATION] ====================
Generated Cypher:
MATCH (c:Certification {title: 'Project Management'})
RETURN count(DISTINCT c)

==================== [DEBUG: EXECUTION] ====================
Context

--- DATA (Execution) ---
### LIVE DATABASE SCHEMA (LLM CONTEXT GUIDE)

**1. GRAPH STRUCTURE (Grouped by Source):**
(:Education)-[:AT_UNIVERSITY]->(:University)
(:Education)-[:IN_MAJOR]->(:Major)
(:Education)-[:FOR_DEGREE]->(:Degree)
(:Experience)-[:AT_COMPANY]->(:Company)
(:Experience)-[:PITCHED_INTO]->(:Experience)
(:Experience)-[:FOLLOWED_BY]->(:Experience)
(:Experience)-[:HAPPENED_DURING]->(:Experience)
(:Experience)-[:ROLE_WAS]->(:JobTitle)
(:Professional)-[:HAS_CERTIFICATION]->(:Certification)
(:Professional)-[:HAS_EDUCATION]->(:Education)
(:Professional)-[:FIRST_INDUSTRY]->(:Industry)
(:P